# P.I.S.T.O.N. Model Training V4

Final Senior Design 2 model pipeline using the finalized V4 dataset and fixed vehicle-based splits.


## 1. Setup and Final Dataset

Load the finalized workbook, features, splits, and model settings.


In [1]:
# ============================================================
# P.I.S.T.O.N. MODEL TRAINING V4
#
# Final Senior Design 2 model pipeline.
#
# Uses the finalized V4 master dataset with fixed
# vehicle-based Train / Validation / Test splits.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

V4_MASTER_FILE = (
    "PISTON_MLV4_MASTERDATASET.xlsx"
)

fuel_cases = pd.read_excel(
    V4_MASTER_FILE,
    sheet_name="FUEL_CASES"
)

catalyst_cases = pd.read_excel(
    V4_MASTER_FILE,
    sheet_name="CATALYST_CASES"
)

charging_cases = pd.read_excel(
    V4_MASTER_FILE,
    sheet_name="CHARGING_CASES"
)

FUEL_FEATURES = [
    "ENGINE_RPM_MEAN",
    "ENGINE_LOAD_PCT_MEAN",
    "SELECTED_BANK_STFT_PCT_MEAN",
    "SELECTED_BANK_STFT_PCT_STD",
    "SELECTED_BANK_LTFT_PCT_MEAN",
    "SELECTED_BANK_LTFT_PCT_STD",
    "SELECTED_BANK_TOTAL_TRIM_PCT_MEAN",
    "SELECTED_BANK_ABS_TOTAL_TRIM_PCT_MEAN",
    "SELECTED_BANK_ABS_TOTAL_TRIM_PCT_MIN",
]

CATALYST_FEATURES = [
    "CAT_MAX_LAGGED_CORR",
    "CAT_DOWNSTREAM_STD",
    "CAT_DOWNSTREAM_RANGE",
    "CAT_DOWNSTREAM_MEAN_ABS_STEP",
    "CAT_DOWNSTREAM_STEP_STD",
]

CHARGING_FEATURES = [
    "CONTROL_MODULE_VOLTAGE_V_MEAN",
    "CONTROL_MODULE_VOLTAGE_V_MIN",
    "CONTROL_MODULE_VOLTAGE_V_MAX",
    "BELOW_CHARGING_COUNT_LT_13_0",
    "ABOVE_CHARGING_COUNT_GT_14_8",
    "VOLTAGE_MAX_ABS_STEP",
    "VOLTAGE_STEP_STD",
]

FUEL_IF_QUANTILE = 0.97

CATALYST_IF_QUANTILE = 0.985
CATALYST_CORRELATION_THRESHOLD = 0.75
CATALYST_REQUIRED_CONSECUTIVE_CASES = 2

CHARGING_IF_QUANTILE = 0.97

print("=" * 100)
print("P.I.S.T.O.N. V4 FINAL DATASET")
print("=" * 100)

print("Fuel cases:", len(fuel_cases))
print("Catalyst cases:", len(catalyst_cases))
print("Charging cases:", len(charging_cases))

print("\nFuel split:")
print(
    fuel_cases["DATA_SPLIT"]
    .value_counts()
    .reindex(["TRAIN","VALIDATION","TEST"])
    .to_string()
)

print("\nCatalyst split:")
print(
    catalyst_cases["DATA_SPLIT"]
    .value_counts()
    .reindex(["TRAIN","VALIDATION","TEST"])
    .to_string()
)

print("\nCharging split:")
print(
    charging_cases["DATA_SPLIT"]
    .value_counts()
    .reindex(["TRAIN","VALIDATION","TEST"])
    .to_string()
)

for subsystem_name, data, features in [
    ("FUEL", fuel_cases, FUEL_FEATURES),
    ("CATALYST", catalyst_cases, CATALYST_FEATURES),
    ("CHARGING", charging_cases, CHARGING_FEATURES),
]:
    missing = [
        feature
        for feature in features
        if feature not in data.columns
    ]
    print(f"\n{subsystem_name} feature count:", len(features))
    print(f"{subsystem_name} missing features:", missing)


P.I.S.T.O.N. V4 FINAL DATASET
Fuel cases: 811
Catalyst cases: 547
Charging cases: 813

Fuel split:
DATA_SPLIT
TRAIN         395
VALIDATION    106
TEST          310

Catalyst split:
DATA_SPLIT
TRAIN         204
VALIDATION    106
TEST          237

Charging split:
DATA_SPLIT
TRAIN         395
VALIDATION    106
TEST          312

FUEL feature count: 9
FUEL missing features: []

CATALYST feature count: 5
CATALYST missing features: []

CHARGING feature count: 7
CHARGING missing features: []


## 2. Fuel Model Training

Train the final 9-feature Fuel Isolation Forest using Normal training cases only.


In [2]:
# ============================================================
# FUEL - FINAL MODEL TRAINING
#
# Train only on Normal TRAIN cases.
#
# TARGET_ABNORMAL represents observed trim behavior,
# not confirmed mechanical-fault ground truth.
# ============================================================


# ============================================================
# PREPARE LABELS
# ============================================================

fuel_cases[
    "TARGET_ABNORMAL"
] = pd.to_numeric(
    fuel_cases[
        "TARGET_ABNORMAL"
    ],
    errors="coerce"
)


# ============================================================
# SPLIT DATA
# ============================================================

fuel_train_normal = (
    fuel_cases[
        (
            fuel_cases[
                "DATA_SPLIT"
            ] == "TRAIN"
        )
        &
        (
            fuel_cases[
                "TARGET_ABNORMAL"
            ] == 0
        )
    ]
    .copy()
)


fuel_validation = (
    fuel_cases[
        fuel_cases[
            "DATA_SPLIT"
        ] == "VALIDATION"
    ]
    .copy()
)


fuel_test = (
    fuel_cases[
        fuel_cases[
            "DATA_SPLIT"
        ] == "TEST"
    ]
    .copy()
)


print("=" * 100)
print("FUEL FINAL MODEL DATA")
print("=" * 100)

print(
    "Normal training cases:",
    len(fuel_train_normal)
)

print(
    "Validation cases:",
    len(fuel_validation)
)

print(
    "Test cases:",
    len(fuel_test)
)


# ============================================================
# IMPUTE FEATURES
# ============================================================

fuel_imputer = SimpleImputer(
    strategy="median"
)


X_fuel_train = (
    fuel_imputer.fit_transform(
        fuel_train_normal[
            FUEL_FEATURES
        ]
    )
)


X_fuel_validation = (
    fuel_imputer.transform(
        fuel_validation[
            FUEL_FEATURES
        ]
    )
)


X_fuel_test = (
    fuel_imputer.transform(
        fuel_test[
            FUEL_FEATURES
        ]
    )
)


# ============================================================
# TRAIN ISOLATION FOREST
# ============================================================

fuel_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)


fuel_model.fit(
    X_fuel_train
)


# higher score = more abnormal
fuel_train_scores = (
    -fuel_model.decision_function(
        X_fuel_train
    )
)


FUEL_IF_THRESHOLD = np.quantile(
    fuel_train_scores,
    FUEL_IF_QUANTILE
)


print(
    "\nFuel IF quantile:",
    FUEL_IF_QUANTILE
)

print(
    "Fuel anomaly-score threshold:",
    round(
        FUEL_IF_THRESHOLD,
        6
    )
)


FUEL FINAL MODEL DATA
Normal training cases: 338
Validation cases: 106
Test cases: 310

Fuel IF quantile: 0.97
Fuel anomaly-score threshold: 0.080853


## 3. Fuel Evaluation

Evaluate the frozen Fuel model on the fixed Validation and Test vehicle splits.


In [3]:
# ============================================================
# FUEL - FINAL HELD-OUT EVALUATION
#
# Evaluate the frozen 0.97 Fuel configuration.
#
# Monitor+ detection is behavior-label detection,
# not confirmed mechanical-fault recall.
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_fuel_split(
    split_name,
    data,
    features
):

    scores = (
        -fuel_model.decision_function(
            features
        )
    )


    predictions = (
        scores
        >= FUEL_IF_THRESHOLD
    ).astype(int)


    actual = (
        data[
            "TARGET_ABNORMAL"
        ]
        .astype(int)
        .to_numpy()
    )


    cm = confusion_matrix(
        actual,
        predictions,
        labels=[
            0,
            1
        ]
    )


    tn, fp, fn, tp = (
        cm.ravel()
    )


    normal_alert_rate = (
        fp
        /
        (
            tn
            +
            fp
        )
        if (
            tn
            +
            fp
        ) > 0
        else np.nan
    )


    results = {
        "SPLIT":
            split_name,

        "CASES":
            len(data),

        "NORMAL_CASES":
            (
                actual == 0
            ).sum(),

        "MONITOR_PLUS_CASES":
            (
                actual == 1
            ).sum(),

        "ACCURACY":
            accuracy_score(
                actual,
                predictions
            ),

        "BALANCED_ACCURACY":
            balanced_accuracy_score(
                actual,
                predictions
            ),

        "PRECISION":
            precision_score(
                actual,
                predictions,
                zero_division=0
            ),

        "MONITOR_PLUS_DETECTION":
            recall_score(
                actual,
                predictions,
                zero_division=0
            ),

        "F1":
            f1_score(
                actual,
                predictions,
                zero_division=0
            ),

        "NORMAL_ALERT_RATE":
            normal_alert_rate,

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp,
    }


    return (
        results,
        scores,
        predictions
    )


# ============================================================
# VALIDATION
# ============================================================

(
    fuel_validation_results,
    fuel_validation_scores,
    fuel_validation_predictions
) = evaluate_fuel_split(
    "VALIDATION",
    fuel_validation,
    X_fuel_validation
)


# ============================================================
# TEST
# ============================================================

(
    fuel_test_results,
    fuel_test_scores,
    fuel_test_predictions
) = evaluate_fuel_split(
    "TEST",
    fuel_test,
    X_fuel_test
)


# ============================================================
# RESULTS
# ============================================================

fuel_final_results = pd.DataFrame(
    [
        fuel_validation_results,
        fuel_test_results
    ]
)


print("\n" + "=" * 125)
print("FUEL HELD-OUT BEHAVIOR-LABEL RESULTS")
print("=" * 125)

print(
    fuel_final_results
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# SAVE SCORES IN MEMORY FOR LATER SUMMARY
# ============================================================

fuel_validation[
    "FUEL_IF_SCORE"
] = fuel_validation_scores


fuel_validation[
    "FUEL_PREDICTED_ABNORMAL"
] = fuel_validation_predictions


fuel_test[
    "FUEL_IF_SCORE"
] = fuel_test_scores


fuel_test[
    "FUEL_PREDICTED_ABNORMAL"
] = fuel_test_predictions



FUEL HELD-OUT BEHAVIOR-LABEL RESULTS
     SPLIT  CASES  NORMAL_CASES  MONITOR_PLUS_CASES  ACCURACY  BALANCED_ACCURACY  PRECISION  MONITOR_PLUS_DETECTION     F1  NORMAL_ALERT_RATE  TN  FP  FN  TP
VALIDATION    106            49                  57    0.9340             0.9300     0.9032                  0.9825 0.9412             0.1224  43   6   1  56
      TEST    310           207                 103    0.8903             0.8691     0.8557                  0.8058 0.8300             0.0676 193  14  20  83


## 4. Catalyst Model and Evaluation

Final Catalyst hybrid model and held-out evaluation.


In [4]:
# ============================================================
# CATALYST - FINAL ONE-CLASS HYBRID MODEL AND EVALUATION
#
# Train the Isolation Forest only on evaluable Normal
# training cases.
#
# Final alert:
# - Isolation Forest anomaly
# OR
# - high correlation for 2 consecutive eligible cases
# ============================================================


# ============================================================
# PREPARE DATA
# ============================================================

catalyst_cases[
    "CATALYST_SIGNAL_EVALUABLE"
] = (
    catalyst_cases[
        "CATALYST_SIGNAL_EVALUABLE"
    ]
    .fillna(False)
    .astype(bool)
)


catalyst_train_normal = (
    catalyst_cases[
        (
            catalyst_cases[
                "DATA_SPLIT"
            ] == "TRAIN"
        )
        &
        (
            catalyst_cases[
                "CASE_BEHAVIOR_LABEL"
            ] == "NORMAL"
        )
        &
        (
            catalyst_cases[
                "CATALYST_SIGNAL_EVALUABLE"
            ] == True
        )
    ]
    .copy()
)


print("=" * 100)
print("CATALYST FINAL MODEL DATA")
print("=" * 100)

print(
    "Normal evaluable training cases:",
    len(catalyst_train_normal)
)

print(
    "Validation cases:",
    (
        catalyst_cases[
            "DATA_SPLIT"
        ] == "VALIDATION"
    ).sum()
)

print(
    "Test cases:",
    (
        catalyst_cases[
            "DATA_SPLIT"
        ] == "TEST"
    ).sum()
)


# ============================================================
# IMPUTE FEATURES
# ============================================================

catalyst_imputer = SimpleImputer(
    strategy="median"
)


X_catalyst_train = (
    catalyst_imputer.fit_transform(
        catalyst_train_normal[
            CATALYST_FEATURES
        ]
    )
)


# ============================================================
# TRAIN ISOLATION FOREST
# ============================================================

catalyst_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)


catalyst_model.fit(
    X_catalyst_train
)


# higher score = more abnormal
catalyst_train_scores = (
    -catalyst_model.decision_function(
        X_catalyst_train
    )
)


CATALYST_IF_THRESHOLD = np.quantile(
    catalyst_train_scores,
    CATALYST_IF_QUANTILE
)


print(
    "\nCatalyst IF quantile:",
    CATALYST_IF_QUANTILE
)

print(
    "Catalyst anomaly-score threshold:",
    round(
        CATALYST_IF_THRESHOLD,
        6
    )
)

print(
    "Correlation threshold:",
    CATALYST_CORRELATION_THRESHOLD
)

print(
    "Required consecutive cases:",
    CATALYST_REQUIRED_CONSECUTIVE_CASES
)


# ============================================================
# SCORE EVALUABLE CASES
# ============================================================

catalyst_cases[
    "CATALYST_IF_SCORE"
] = np.nan


catalyst_cases[
    "CATALYST_IF_ALERT"
] = False


evaluable_mask = (
    catalyst_cases[
        "CATALYST_SIGNAL_EVALUABLE"
    ] == True
)


X_catalyst_evaluable = (
    catalyst_imputer.transform(
        catalyst_cases.loc[
            evaluable_mask,
            CATALYST_FEATURES
        ]
    )
)


catalyst_scores = (
    -catalyst_model.decision_function(
        X_catalyst_evaluable
    )
)


catalyst_cases.loc[
    evaluable_mask,
    "CATALYST_IF_SCORE"
] = catalyst_scores


catalyst_cases.loc[
    evaluable_mask,
    "CATALYST_IF_ALERT"
] = (
    catalyst_scores
    >= CATALYST_IF_THRESHOLD
)


# ============================================================
# PERSISTENT MIRRORING
# ============================================================

catalyst_cases[
    "CATALYST_PERSISTENT_MIRROR"
] = False


order_column = None

for candidate in [
    "START_SAMPLE",
    "START_ROW",
    "CASE_NUMBER",
]:

    if candidate in catalyst_cases.columns:

        order_column = candidate
        break


if "SOURCE_FILE" not in catalyst_cases.columns:

    raise ValueError(
        "SOURCE_FILE is required for Catalyst sequence checking."
    )


working_catalyst = (
    catalyst_cases
    .reset_index()
    .rename(
        columns={
            "index":
                "ORIGINAL_INDEX"
        }
    )
)


sort_columns = [
    "SOURCE_FILE"
]


if order_column is not None:

    sort_columns.append(
        order_column
    )


working_catalyst = (
    working_catalyst
    .sort_values(
        sort_columns
    )
)


for source_file, group in working_catalyst.groupby(
    "SOURCE_FILE",
    dropna=False
):

    streak = 0


    for _, row in group.iterrows():

        original_index = int(
            row[
                "ORIGINAL_INDEX"
            ]
        )


        correlation = pd.to_numeric(
            row[
                "CAT_MAX_LAGGED_CORR"
            ],
            errors="coerce"
        )


        high_correlation = (
            bool(
                row[
                    "CATALYST_SIGNAL_EVALUABLE"
                ]
            )
            and
            pd.notna(
                correlation
            )
            and
            correlation
            >= CATALYST_CORRELATION_THRESHOLD
        )


        if high_correlation:

            streak += 1

        else:

            streak = 0


        if (
            streak
            >=
            CATALYST_REQUIRED_CONSECUTIVE_CASES
        ):

            catalyst_cases.loc[
                original_index,
                "CATALYST_PERSISTENT_MIRROR"
            ] = True


# ============================================================
# FINAL HYBRID ALERT
# ============================================================

catalyst_cases[
    "CATALYST_HYBRID_ALERT"
] = (
    catalyst_cases[
        "CATALYST_SIGNAL_EVALUABLE"
    ]
    &
    (
        catalyst_cases[
            "CATALYST_IF_ALERT"
        ]
        |
        catalyst_cases[
            "CATALYST_PERSISTENT_MIRROR"
        ]
    )
)


# ============================================================
# HELD-OUT NORMAL RESULTS
# ============================================================

def catalyst_normal_results(
    split_name
):

    data = (
        catalyst_cases[
            (
                catalyst_cases[
                    "DATA_SPLIT"
                ] == split_name
            )
            &
            (
                catalyst_cases[
                    "CASE_BEHAVIOR_LABEL"
                ] == "NORMAL"
            )
            &
            (
                catalyst_cases[
                    "CATALYST_SIGNAL_EVALUABLE"
                ] == True
            )
        ]
        .copy()
    )


    return {
        "SPLIT":
            split_name,

        "EVALUABLE_NORMAL_CASES":
            len(data),

        "IF_ALERTS":
            int(
                data[
                    "CATALYST_IF_ALERT"
                ].sum()
            ),

        "PERSISTENT_MIRROR_CASES":
            int(
                data[
                    "CATALYST_PERSISTENT_MIRROR"
                ].sum()
            ),

        "HYBRID_ALERTS":
            int(
                data[
                    "CATALYST_HYBRID_ALERT"
                ].sum()
            ),

        "NORMAL_ALERT_RATE":
            (
                data[
                    "CATALYST_HYBRID_ALERT"
                ].mean()
                if len(data) > 0
                else np.nan
            ),
    }


catalyst_normal_summary = pd.DataFrame(
    [
        catalyst_normal_results(
            "VALIDATION"
        ),

        catalyst_normal_results(
            "TEST"
        ),
    ]
)


print("\n" + "=" * 110)
print("CATALYST HELD-OUT NORMAL RESULTS")
print("=" * 110)

print(
    catalyst_normal_summary
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# CONFIRMED-DTC RECORDING EVIDENCE
#
# These cases remain Uncertain at the individual case level.
# This reports diagnostic evidence, not abnormal recall.
# ============================================================

catalyst_uncertain = (
    catalyst_cases[
        (
            catalyst_cases[
                "DATA_SPLIT"
            ].isin(
                [
                    "VALIDATION",
                    "TEST",
                ]
            )
        )
        &
        (
            catalyst_cases[
                "CASE_BEHAVIOR_LABEL"
            ] == "UNCERTAIN"
        )
    ]
    .copy()
)


vehicle_column = (
    "VEHICLE_ID"
    if "VEHICLE_ID" in catalyst_cases.columns
    else "SOURCE_FILE"
)


catalyst_dtc_rows = []


for vehicle, data in catalyst_uncertain.groupby(
    vehicle_column,
    dropna=False
):

    catalyst_dtc_rows.append(
        {
            "VEHICLE":
                vehicle,

            "SPLIT":
                data[
                    "DATA_SPLIT"
                ].iloc[0],

            "TOTAL_CASES":
                len(data),

            "EVALUABLE_CASES":
                int(
                    data[
                        "CATALYST_SIGNAL_EVALUABLE"
                    ].sum()
                ),

            "IF_ALERT_CASES":
                int(
                    data[
                        "CATALYST_IF_ALERT"
                    ].sum()
                ),

            "PERSISTENT_MIRROR_CASES":
                int(
                    data[
                        "CATALYST_PERSISTENT_MIRROR"
                    ].sum()
                ),

            "HYBRID_EVIDENCE_CASES":
                int(
                    data[
                        "CATALYST_HYBRID_ALERT"
                    ].sum()
                ),
        }
    )


catalyst_dtc_summary = pd.DataFrame(
    catalyst_dtc_rows
)


print("\n" + "=" * 110)
print("CATALYST CONFIRMED-DTC RECORDING EVIDENCE")
print("=" * 110)

print(
    catalyst_dtc_summary
    .to_string(
        index=False
    )
)

CATALYST FINAL MODEL DATA
Normal evaluable training cases: 89
Validation cases: 106
Test cases: 237

Catalyst IF quantile: 0.985
Catalyst anomaly-score threshold: 0.113591
Correlation threshold: 0.75
Required consecutive cases: 2


C:\Users\yoboy\AppData\Local\Temp\ipykernel_37936\2887103829.py:158: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  catalyst_cases[
C:\Users\yoboy\AppData\Local\Temp\ipykernel_37936\2887103829.py:163: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  catalyst_cases[
C:\Users\yoboy\AppData\Local\Temp\ipykernel_37936\2887103829.py:211: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axi


CATALYST HELD-OUT NORMAL RESULTS
     SPLIT  EVALUABLE_NORMAL_CASES  IF_ALERTS  PERSISTENT_MIRROR_CASES  HYBRID_ALERTS  NORMAL_ALERT_RATE
VALIDATION                      13          1                        0              1             0.0769
      TEST                     201          7                        6             12             0.0597

CATALYST CONFIRMED-DTC RECORDING EVIDENCE
      VEHICLE      SPLIT  TOTAL_CASES  EVALUABLE_CASES  IF_ALERT_CASES  PERSISTENT_MIRROR_CASES  HYBRID_EVIDENCE_CASES
   CRUZE_2013 VALIDATION           93               93               2                        0                      2
EXPLORER_2015       TEST           26                0               0                        0                      0


## 5. Charging Model and Evaluation

Final Charging model and held-out evaluation.


In [5]:
# ============================================================
# CHARGING - FINAL MODEL AND EVALUATION
#
# Train only on Normal TRAIN cases.
#
# TARGET_ABNORMAL represents observed charging behavior,
# not confirmed charging-system fault ground truth.
# ============================================================


# ============================================================
# PREPARE LABELS
# ============================================================

charging_cases[
    "TARGET_ABNORMAL"
] = pd.to_numeric(
    charging_cases[
        "TARGET_ABNORMAL"
    ],
    errors="coerce"
)


# ============================================================
# SPLIT DATA
# ============================================================

charging_train_normal = (
    charging_cases[
        (
            charging_cases[
                "DATA_SPLIT"
            ] == "TRAIN"
        )
        &
        (
            charging_cases[
                "TARGET_ABNORMAL"
            ] == 0
        )
    ]
    .copy()
)


charging_validation = (
    charging_cases[
        charging_cases[
            "DATA_SPLIT"
        ] == "VALIDATION"
    ]
    .copy()
)


charging_test = (
    charging_cases[
        charging_cases[
            "DATA_SPLIT"
        ] == "TEST"
    ]
    .copy()
)


print("=" * 100)
print("CHARGING FINAL MODEL DATA")
print("=" * 100)

print(
    "Normal training cases:",
    len(charging_train_normal)
)

print(
    "Validation cases:",
    len(charging_validation)
)

print(
    "Test cases:",
    len(charging_test)
)


# ============================================================
# IMPUTE FEATURES
# ============================================================

charging_imputer = SimpleImputer(
    strategy="median"
)


X_charging_train = (
    charging_imputer.fit_transform(
        charging_train_normal[
            CHARGING_FEATURES
        ]
    )
)


X_charging_validation = (
    charging_imputer.transform(
        charging_validation[
            CHARGING_FEATURES
        ]
    )
)


X_charging_test = (
    charging_imputer.transform(
        charging_test[
            CHARGING_FEATURES
        ]
    )
)


# ============================================================
# TRAIN ISOLATION FOREST
# ============================================================

charging_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)


charging_model.fit(
    X_charging_train
)


# higher score = more abnormal
charging_train_scores = (
    -charging_model.decision_function(
        X_charging_train
    )
)


CHARGING_IF_THRESHOLD = np.quantile(
    charging_train_scores,
    CHARGING_IF_QUANTILE
)


print(
    "\nCharging IF quantile:",
    CHARGING_IF_QUANTILE
)

print(
    "Charging anomaly-score threshold:",
    round(
        CHARGING_IF_THRESHOLD,
        6
    )
)


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_charging_split(
    split_name,
    data,
    features
):

    scores = (
        -charging_model.decision_function(
            features
        )
    )


    predictions = (
        scores
        >= CHARGING_IF_THRESHOLD
    ).astype(int)


    actual = (
        data[
            "TARGET_ABNORMAL"
        ]
        .astype(int)
        .to_numpy()
    )


    cm = confusion_matrix(
        actual,
        predictions,
        labels=[
            0,
            1
        ]
    )


    tn, fp, fn, tp = (
        cm.ravel()
    )


    normal_alert_rate = (
        fp
        /
        (
            tn
            +
            fp
        )
        if (
            tn
            +
            fp
        ) > 0
        else np.nan
    )


    monitor_plus_detection = (
        tp
        /
        (
            tp
            +
            fn
        )
        if (
            tp
            +
            fn
        ) > 0
        else np.nan
    )


    return {
        "SPLIT":
            split_name,

        "CASES":
            len(data),

        "NORMAL_CASES":
            (
                actual == 0
            ).sum(),

        "MONITOR_PLUS_CASES":
            (
                actual == 1
            ).sum(),

        "ACCURACY":
            accuracy_score(
                actual,
                predictions
            ),

        "BALANCED_ACCURACY":
            balanced_accuracy_score(
                actual,
                predictions
            ),

        "PRECISION":
            precision_score(
                actual,
                predictions,
                zero_division=0
            ),

        "MONITOR_PLUS_DETECTION":
            monitor_plus_detection,

        "F1":
            f1_score(
                actual,
                predictions,
                zero_division=0
            ),

        "NORMAL_ALERT_RATE":
            normal_alert_rate,

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp,
    }


# ============================================================
# VALIDATION + TEST
# ============================================================

charging_validation_results = (
    evaluate_charging_split(
        "VALIDATION",
        charging_validation,
        X_charging_validation
    )
)


charging_test_results = (
    evaluate_charging_split(
        "TEST",
        charging_test,
        X_charging_test
    )
)


charging_final_results = pd.DataFrame(
    [
        charging_validation_results,
        charging_test_results
    ]
)


print("\n" + "=" * 125)
print("CHARGING HELD-OUT BEHAVIOR-LABEL RESULTS")
print("=" * 125)

print(
    charging_final_results
    .round(4)
    .to_string(
        index=False
    )
)

CHARGING FINAL MODEL DATA
Normal training cases: 327
Validation cases: 106
Test cases: 312

Charging IF quantile: 0.97
Charging anomaly-score threshold: 0.146576

CHARGING HELD-OUT BEHAVIOR-LABEL RESULTS
     SPLIT  CASES  NORMAL_CASES  MONITOR_PLUS_CASES  ACCURACY  BALANCED_ACCURACY  PRECISION  MONITOR_PLUS_DETECTION     F1  NORMAL_ALERT_RATE  TN  FP  FN  TP
VALIDATION    106           105                   1    1.0000             1.0000     1.0000                  1.0000 1.0000             0.0000 105   0   0   1
      TEST    312           209                 103    0.9199             0.9328     0.8197                  0.9709 0.8889             0.1053 187  22   3 100


## 6. Controlled Challenge Results

Controlled subsystem challenge-case evaluation.


In [6]:
# ============================================================
# CONTROLLED CHALLENGE RESULTS
#
# These results were produced during SD2 development using
# controlled challenge cases.
#
# They are used to test expected abnormal behavior and are
# not treated as confirmed real-world fault accuracy.
# ============================================================


# ============================================================
# FUEL CONTROLLED CHALLENGE
# ============================================================

fuel_challenge_results = pd.DataFrame(
    [
        {
            "CHALLENGE": "Lean Mild",
            "DETECTION_RATE": 0.7729,
        },
        {
            "CHALLENGE": "Lean Moderate",
            "DETECTION_RATE": 0.9890,
        },
        {
            "CHALLENGE": "Lean Strong",
            "DETECTION_RATE": 1.0000,
        },
        {
            "CHALLENGE": "Rich Mild",
            "DETECTION_RATE": 0.8791,
        },
        {
            "CHALLENGE": "Rich Moderate",
            "DETECTION_RATE": 0.9853,
        },
        {
            "CHALLENGE": "Rich Strong",
            "DETECTION_RATE": 1.0000,
        },
    ]
)


print("=" * 90)
print("FUEL CONTROLLED CHALLENGE RESULTS")
print("=" * 90)

print(
    fuel_challenge_results
    .to_string(
        index=False
    )
)


print(
    "\nOverall controlled abnormal detection:",
    round(
        0.9377,
        4
    )
)

print(
    "Moderate + Strong detection:",
    round(
        0.9936,
        4
    )
)


# ============================================================
# CATALYST CONTROLLED CHALLENGE
# ============================================================

catalyst_challenge_results = pd.DataFrame(
    [
        {
            "CHALLENGE": "Mild",
            "DETECTION_RATE": 0.0660,
        },
        {
            "CHALLENGE": "Moderate",
            "DETECTION_RATE": 0.9637,
        },
        {
            "CHALLENGE": "Strong",
            "DETECTION_RATE": 0.9802,
        },
    ]
)


print("\n" + "=" * 90)
print("CATALYST CONTROLLED CHALLENGE RESULTS")
print("=" * 90)

print(
    catalyst_challenge_results
    .to_string(
        index=False
    )
)


print(
    "\nModerate + Strong detection:",
    round(
        0.9719,
        4
    )
)


# ============================================================
# CHARGING CONTROLLED CHALLENGE
# ============================================================

charging_challenge_results = pd.DataFrame(
    [
        {
            "CHALLENGE": "Low Voltage",
            "DETECTED": 30,
            "TOTAL": 30,
            "DETECTION_RATE": 1.0000,
        },
        {
            "CHALLENGE": "High Voltage",
            "DETECTED": 30,
            "TOTAL": 30,
            "DETECTION_RATE": 1.0000,
        },
        {
            "CHALLENGE": "Unstable Voltage",
            "DETECTED": 30,
            "TOTAL": 30,
            "DETECTION_RATE": 1.0000,
        },
    ]
)


print("\n" + "=" * 90)
print("CHARGING CONTROLLED CHALLENGE RESULTS")
print("=" * 90)

print(
    charging_challenge_results
    .to_string(
        index=False
    )
)


print(
    "\nOverall controlled challenge detection:",
    round(
        1.0000,
        4
    )
)

FUEL CONTROLLED CHALLENGE RESULTS
    CHALLENGE  DETECTION_RATE
    Lean Mild          0.7729
Lean Moderate          0.9890
  Lean Strong          1.0000
    Rich Mild          0.8791
Rich Moderate          0.9853
  Rich Strong          1.0000

Overall controlled abnormal detection: 0.9377
Moderate + Strong detection: 0.9936

CATALYST CONTROLLED CHALLENGE RESULTS
CHALLENGE  DETECTION_RATE
     Mild          0.0660
 Moderate          0.9637
   Strong          0.9802

Moderate + Strong detection: 0.9719

CHARGING CONTROLLED CHALLENGE RESULTS
       CHALLENGE  DETECTED  TOTAL  DETECTION_RATE
     Low Voltage        30     30             1.0
    High Voltage        30     30             1.0
Unstable Voltage        30     30             1.0

Overall controlled challenge detection: 1.0


## 7. Final Summary and Configuration

Final model settings and subsystem results.


In [7]:
# ============================================================
# FINAL SUMMARY AND CONFIGURATION
#
# Summarize the final models, features, thresholds,
# held-out results, and controlled challenge results.
# ============================================================


# ============================================================
# TEST RESULTS
# ============================================================

fuel_test_row = (
    fuel_final_results[
        fuel_final_results[
            "SPLIT"
        ] == "TEST"
    ]
    .iloc[0]
)


charging_test_row = (
    charging_final_results[
        charging_final_results[
            "SPLIT"
        ] == "TEST"
    ]
    .iloc[0]
)


catalyst_test_row = (
    catalyst_normal_summary[
        catalyst_normal_summary[
            "SPLIT"
        ] == "TEST"
    ]
    .iloc[0]
)


final_results = pd.DataFrame(
    [
        {
            "SUBSYSTEM":
                "Fuel",

            "FINAL_METHOD":
                "Isolation Forest",

            "FEATURES":
                len(FUEL_FEATURES),

            "TEST_ACCURACY":
                fuel_test_row[
                    "ACCURACY"
                ],

            "TEST_BEHAVIOR_DETECTION":
                fuel_test_row[
                    "MONITOR_PLUS_DETECTION"
                ],

            "TEST_NORMAL_ALERT_RATE":
                fuel_test_row[
                    "NORMAL_ALERT_RATE"
                ],

            "CONTROLLED_CHALLENGE":
                0.9377,

            "MODERATE_STRONG_CHALLENGE":
                0.9936,
        },

        {
            "SUBSYSTEM":
                "Catalyst",

            "FINAL_METHOD":
                "One-Class IF + persistent mirroring",

            "FEATURES":
                len(CATALYST_FEATURES),

            "TEST_ACCURACY":
                np.nan,

            "TEST_BEHAVIOR_DETECTION":
                np.nan,

            "TEST_NORMAL_ALERT_RATE":
                catalyst_test_row[
                    "NORMAL_ALERT_RATE"
                ],

            "CONTROLLED_CHALLENGE":
                0.6700,

            "MODERATE_STRONG_CHALLENGE":
                0.9719,
        },

        {
            "SUBSYSTEM":
                "Charging",

            "FINAL_METHOD":
                "Isolation Forest",

            "FEATURES":
                len(CHARGING_FEATURES),

            "TEST_ACCURACY":
                charging_test_row[
                    "ACCURACY"
                ],

            "TEST_BEHAVIOR_DETECTION":
                charging_test_row[
                    "MONITOR_PLUS_DETECTION"
                ],

            "TEST_NORMAL_ALERT_RATE":
                charging_test_row[
                    "NORMAL_ALERT_RATE"
                ],

            "CONTROLLED_CHALLENGE":
                1.0000,

            "MODERATE_STRONG_CHALLENGE":
                np.nan,
        },
    ]
)


print("=" * 125)
print("P.I.S.T.O.N. FINAL V4 RESULTS")
print("=" * 125)

print(
    final_results
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# FINAL MODEL SETTINGS
# ============================================================

final_configuration = pd.DataFrame(
    [
        {
            "SUBSYSTEM":
                "Fuel",

            "FEATURE_COUNT":
                len(FUEL_FEATURES),

            "IF_QUANTILE":
                FUEL_IF_QUANTILE,

            "IF_THRESHOLD":
                FUEL_IF_THRESHOLD,

            "CORRELATION_THRESHOLD":
                np.nan,

            "CONSECUTIVE_CASES":
                np.nan,
        },

        {
            "SUBSYSTEM":
                "Catalyst",

            "FEATURE_COUNT":
                len(CATALYST_FEATURES),

            "IF_QUANTILE":
                CATALYST_IF_QUANTILE,

            "IF_THRESHOLD":
                CATALYST_IF_THRESHOLD,

            "CORRELATION_THRESHOLD":
                CATALYST_CORRELATION_THRESHOLD,

            "CONSECUTIVE_CASES":
                CATALYST_REQUIRED_CONSECUTIVE_CASES,
        },

        {
            "SUBSYSTEM":
                "Charging",

            "FEATURE_COUNT":
                len(CHARGING_FEATURES),

            "IF_QUANTILE":
                CHARGING_IF_QUANTILE,

            "IF_THRESHOLD":
                CHARGING_IF_THRESHOLD,

            "CORRELATION_THRESHOLD":
                np.nan,

            "CONSECUTIVE_CASES":
                np.nan,
        },
    ]
)


print("\n" + "=" * 125)
print("P.I.S.T.O.N. FINAL MODEL CONFIGURATION")
print("=" * 125)

print(
    final_configuration
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# FINAL FEATURE SETS
# ============================================================

print("\n" + "=" * 125)
print("FINAL FEATURE SETS")
print("=" * 125)


print("\nFuel:")
for feature in FUEL_FEATURES:
    print(
        "-",
        feature
    )


print("\nCatalyst:")
for feature in CATALYST_FEATURES:
    print(
        "-",
        feature
    )


print("\nCharging:")
for feature in CHARGING_FEATURES:
    print(
        "-",
        feature
    )


# ============================================================
# NOTES
# ============================================================

print("\n" + "=" * 125)
print("FINAL NOTES")
print("=" * 125)

print(
    "- Fuel and Charging behavior detection uses engineering behavior labels,"
    " not confirmed mechanical-fault recall."
)

print(
    "- Catalyst confirmed-DTC cases remain Uncertain at the individual case level."
)

print(
    "- Catalyst controlled Moderate and Strong cases are the useful reduced-efficiency challenge range."
)

print(
    "- Controlled challenge results are simulated development tests,"
    " not real-world fault accuracy."
)

P.I.S.T.O.N. FINAL V4 RESULTS
SUBSYSTEM                        FINAL_METHOD  FEATURES  TEST_ACCURACY  TEST_BEHAVIOR_DETECTION  TEST_NORMAL_ALERT_RATE  CONTROLLED_CHALLENGE  MODERATE_STRONG_CHALLENGE
     Fuel                    Isolation Forest         9         0.8903                   0.8058                  0.0676                0.9377                     0.9936
 Catalyst One-Class IF + persistent mirroring         5            NaN                      NaN                  0.0597                0.6700                     0.9719
 Charging                    Isolation Forest         7         0.9199                   0.9709                  0.1053                1.0000                        NaN

P.I.S.T.O.N. FINAL MODEL CONFIGURATION
SUBSYSTEM  FEATURE_COUNT  IF_QUANTILE  IF_THRESHOLD  CORRELATION_THRESHOLD  CONSECUTIVE_CASES
     Fuel              9        0.970      0.080853                    NaN                NaN
 Catalyst              5        0.985      0.113591               

## Final Model Performance

Summary of the final held-out results and controlled challenge performance for each subsystem.

In [12]:
# ============================================================
# FINAL MODEL PERFORMANCE DISPLAY
#
# Presentation-style summary of the final V4 results.
# ============================================================

from IPython.display import display, HTML


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def format_metric(value):

    if pd.isna(value):
        return "N/A"

    return f"{value:.4f}"


def metric_row(label, value):

    return f"""
    <tr>
        <td class="metric-label">{label}</td>
        <td class="metric-value">{value}</td>
    </tr>
    """


def confusion_table(
    tn,
    fp,
    fn,
    tp
):

    return f"""
    <table class="confusion">
        <tr>
            <th></th>
            <th colspan="2">Predicted</th>
        </tr>
        <tr>
            <th>Actual</th>
            <th>Normal</th>
            <th>Abnormal</th>
        </tr>
        <tr>
            <th>Normal</th>
            <td>{tn}</td>
            <td>{fp}</td>
        </tr>
        <tr>
            <th>Abnormal</th>
            <td>{fn}</td>
            <td>{tp}</td>
        </tr>
    </table>
    """


# ============================================================
# FUEL
# ============================================================

fuel_test = (
    fuel_final_results[
        fuel_final_results[
            "SPLIT"
        ] == "TEST"
    ]
    .iloc[0]
)


fuel_confusion = confusion_table(
    int(fuel_test["TN"]),
    int(fuel_test["FP"]),
    int(fuel_test["FN"]),
    int(fuel_test["TP"])
)


fuel_metrics = (
    metric_row(
        "Accuracy:",
        format_metric(
            fuel_test[
                "ACCURACY"
            ]
        )
    )
    +
    metric_row(
        "Balanced Accuracy:",
        format_metric(
            fuel_test[
                "BALANCED_ACCURACY"
            ]
        )
    )
    +
    metric_row(
        "Precision:",
        format_metric(
            fuel_test[
                "PRECISION"
            ]
        )
    )
    +
    metric_row(
        "Monitor+ Behavior Detection:",
        format_metric(
            fuel_test[
                "MONITOR_PLUS_DETECTION"
            ]
        )
    )
    +
    metric_row(
        "F1 Score:",
        format_metric(
            fuel_test[
                "F1"
            ]
        )
    )
    +
    metric_row(
        "Normal Alert Rate:",
        format_metric(
            fuel_test[
                "NORMAL_ALERT_RATE"
            ]
        )
    )
    +
    metric_row(
        "Controlled Challenge Detection:",
        "0.9377"
    )
    +
    metric_row(
        "Moderate + Strong Challenge Detection:",
        "0.9936"
    )
)


# ============================================================
# CATALYST
# ============================================================

catalyst_test = (
    catalyst_normal_summary[
        catalyst_normal_summary[
            "SPLIT"
        ] == "TEST"
    ]
    .iloc[0]
)


catalyst_normal_cases = int(
    catalyst_test[
        "EVALUABLE_NORMAL_CASES"
    ]
)


catalyst_alerts = int(
    catalyst_test[
        "HYBRID_ALERTS"
    ]
)


catalyst_correct_normal = (
    catalyst_normal_cases
    -
    catalyst_alerts
)


catalyst_metrics = (
    metric_row(
        "Test Normal Alert Rate:",
        format_metric(
            catalyst_test[
                "NORMAL_ALERT_RATE"
            ]
        )
    )
    +
    metric_row(
        "Evaluable Normal Test Cases:",
        str(
            catalyst_normal_cases
        )
    )
    +
    metric_row(
        "Controlled Mild Detection:",
        "0.0660"
    )
    +
    metric_row(
        "Controlled Moderate Detection:",
        "0.9637"
    )
    +
    metric_row(
        "Controlled Strong Detection:",
        "0.9802"
    )
    +
    metric_row(
        "Moderate + Strong Challenge Detection:",
        "0.9719"
    )
)


catalyst_normal_table = f"""
<table class="confusion">
    <tr>
        <th></th>
        <th colspan="2">Hybrid Result</th>
    </tr>
    <tr>
        <th>Actual</th>
        <th>No Alert</th>
        <th>Alert</th>
    </tr>
    <tr>
        <th>Normal</th>
        <td>{catalyst_correct_normal}</td>
        <td>{catalyst_alerts}</td>
    </tr>
</table>
"""


# ============================================================
# CHARGING
# ============================================================

charging_test = (
    charging_final_results[
        charging_final_results[
            "SPLIT"
        ] == "TEST"
    ]
    .iloc[0]
)


charging_confusion = confusion_table(
    int(charging_test["TN"]),
    int(charging_test["FP"]),
    int(charging_test["FN"]),
    int(charging_test["TP"])
)


charging_metrics = (
    metric_row(
        "Accuracy:",
        format_metric(
            charging_test[
                "ACCURACY"
            ]
        )
    )
    +
    metric_row(
        "Balanced Accuracy:",
        format_metric(
            charging_test[
                "BALANCED_ACCURACY"
            ]
        )
    )
    +
    metric_row(
        "Precision:",
        format_metric(
            charging_test[
                "PRECISION"
            ]
        )
    )
    +
    metric_row(
        "Monitor+ Behavior Detection:",
        format_metric(
            charging_test[
                "MONITOR_PLUS_DETECTION"
            ]
        )
    )
    +
    metric_row(
        "F1 Score:",
        format_metric(
            charging_test[
                "F1"
            ]
        )
    )
    +
    metric_row(
        "Normal Alert Rate:",
        format_metric(
            charging_test[
                "NORMAL_ALERT_RATE"
            ]
        )
    )
    +
    metric_row(
        "Controlled Low Voltage Detection:",
        "1.0000"
    )
    +
    metric_row(
        "Controlled High Voltage Detection:",
        "1.0000"
    )
    +
    metric_row(
        "Controlled Unstable Voltage Detection:",
        "1.0000"
    )
)


# ============================================================
# HTML DISPLAY
# ============================================================

html = f"""
<style>

.performance-card {{
    border: 2px solid #617384;
    border-radius: 7px;
    margin: 18px 0;
    overflow: hidden;
    font-family: Arial, sans-serif;
    background: white;
}}

.performance-header {{
    background: #07385d;
    color: white;
    padding: 14px 16px;
}}

.performance-header h2 {{
    margin: 0 0 10px 0;
    font-size: 20px;
}}

.performance-header p {{
    margin: 4px 0;
    font-size: 13px;
}}

.performance-body {{
    padding: 15px 18px 20px 18px;
}}

.metrics {{
    width: 100%;
    border-collapse: collapse;
    margin-bottom: 18px;
}}

.metrics td {{
    padding: 5px 12px;
    border-bottom: 1px solid #eeeeee;
}}

.metrics tr:nth-child(even) {{
    background: #f5f5f5;
}}

.metric-label {{
    text-align: right;
    font-weight: 600;
    width: 70%;
}}

.metric-value {{
    text-align: right;
    width: 30%;
}}

.confusion {{
    border-collapse: collapse;
    margin-top: 8px;
}}

.confusion th,
.confusion td {{
    border: 1px solid #777777;
    padding: 7px 14px;
    text-align: center;
}}

.confusion th {{
    background: #eeeeee;
    font-weight: 600;
}}

.section-label {{
    font-weight: 700;
    margin: 12px 0 5px 0;
}}

.note {{
    margin-top: 15px;
    padding: 8px 10px;
    background: #f6f6f6;
    border-left: 5px solid #e8c547;
    font-size: 12px;
}}

</style>


<div class="performance-card">

    <div class="performance-header">

        <h2>Fuel Delivery</h2>

        <p>
            <b>Final Method:</b>
            Isolation Forest using the selected fuel-trim bank
        </p>

        <p>
            <b>Held-Out Test Cases:</b>
            {int(fuel_test["CASES"])}
        </p>

    </div>

    <div class="performance-body">

        <table class="metrics">
            {fuel_metrics}
        </table>

        <div class="section-label">
            Confusion Matrix
        </div>

        {fuel_confusion}

        <div class="note">
            Monitor+ detection is based on engineering fuel-trim
            behavior labels, not confirmed mechanical-fault recall.
        </div>

    </div>

</div>


<div class="performance-card">

    <div class="performance-header">

        <h2>Catalytic Converter</h2>

        <p>
            <b>Final Method:</b>
            Isolation Forest with persistent upstream/downstream
            O2 mirroring
        </p>

        <p>
            <b>Held-Out Test Cases:</b>
            237
        </p>

    </div>

    <div class="performance-body">

        <table class="metrics">
            {catalyst_metrics}
        </table>

        <div class="section-label">
            Held-Out Evaluable Normal Cases
        </div>

        {catalyst_normal_table}

        <div class="note">
            Confirmed P0420/P0430 recordings are used as supporting
            real-world validation when the required O2 signals are
            diagnostically evaluable. Controlled Moderate and Strong
            cases are used to measure reduced-efficiency detection.
        </div>

    </div>

</div>


<div class="performance-card">

    <div class="performance-header">

        <h2>Charging System</h2>

        <p>
            <b>Final Method:</b>
            Isolation Forest
        </p>

        <p>
            <b>Held-Out Test Cases:</b>
            {int(charging_test["CASES"])}
        </p>

    </div>

    <div class="performance-body">

        <table class="metrics">
            {charging_metrics}
        </table>

        <div class="section-label">
            Confusion Matrix
        </div>

        {charging_confusion}

        <div class="note">
            Monitor+ detection is based on engineering charging
            behavior labels. Controlled Low, High, and Unstable
            voltage challenges are simulated development tests.
        </div>

    </div>

</div>
"""


display(
    HTML(
        html
    )
)

## 8. ESP32 Model Export

Export the final V4 models, settings, and test vectors for embedded implementation.

In [9]:
# ============================================================
# ESP32 MODEL EXPORT
#
# Exports the final V4 models and test vectors for
# implementation and verification on the ESP32.
# ============================================================

from pathlib import Path
import json
import math


EXPORT_DIR = Path("ESP32_Model_Export")
EXPORT_DIR.mkdir(exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def cpp_float(value):

    text = f"{float(value):.9g}"

    # C++ needs a decimal point for whole-number float literals
    if "." not in text and "e" not in text.lower():
        text += ".0"

    return text + "f"


def average_path_length(sample_count):

    if sample_count <= 1:
        return 0.0

    if sample_count == 2:
        return 1.0

    euler_gamma = 0.5772156649015329

    return (
        2.0
        * (
            math.log(sample_count - 1.0)
            + euler_gamma
        )
        -
        2.0
        * (sample_count - 1.0)
        / sample_count
    )


def get_node_depths(tree):

    depths = [0] * tree.node_count
    stack = [(0, 0)]

    while stack:

        node_index, depth = stack.pop()

        depths[node_index] = depth

        left = int(
            tree.children_left[node_index]
        )

        right = int(
            tree.children_right[node_index]
        )

        if left != -1:
            stack.append(
                (left, depth + 1)
            )

        if right != -1:
            stack.append(
                (right, depth + 1)
            )

    return depths


def flatten_isolation_forest(model):

    flat_nodes = []
    tree_offsets = []

    for estimator in model.estimators_:

        tree = estimator.tree_

        tree_offsets.append(
            len(flat_nodes)
        )

        depths = get_node_depths(
            tree
        )

        for node_index in range(
            tree.node_count
        ):

            left = int(
                tree.children_left[node_index]
            )

            right = int(
                tree.children_right[node_index]
            )

            # leaf
            if left == -1 and right == -1:

                leaf_samples = int(
                    tree.n_node_samples[
                        node_index
                    ]
                )

                leaf_path = (
                    depths[node_index]
                    +
                    average_path_length(
                        leaf_samples
                    )
                )

                flat_nodes.append(
                    {
                        "left": -1,
                        "right": -1,
                        "feature": -1,
                        "threshold": 0.0,
                        "leaf_path": leaf_path,
                    }
                )

            # decision node
            else:

                flat_nodes.append(
                    {
                        "left": left,
                        "right": right,
                        "feature": int(
                            tree.feature[
                                node_index
                            ]
                        ),
                        "threshold": float(
                            tree.threshold[
                                node_index
                            ]
                        ),
                        "leaf_path": 0.0,
                    }
                )

    return flat_nodes, tree_offsets


# ============================================================
# SHARED C++ ISOLATION FOREST RUNTIME
# ============================================================

runtime_header = r'''#pragma once

#include <Arduino.h>
#include <math.h>
#include <stdint.h>

struct PistonIFNode
{
    int16_t left;
    int16_t right;
    int8_t feature;
    float threshold;
    float leafPath;
};


// Returns the same score definition used in V4:
// -model.decision_function(X)
//
// Higher score = more abnormal.
static inline float pistonIsolationForestScore(
    const PistonIFNode* nodes,
    const uint32_t* treeOffsets,
    uint16_t treeCount,
    float normalization,
    float modelOffset,
    const float* featureValues,
    const float* imputerMedians)
{
    float depthSum = 0.0f;

    for (
        uint16_t treeIndex = 0;
        treeIndex < treeCount;
        treeIndex++
    )
    {
        uint32_t treeStart =
            treeOffsets[treeIndex];

        int16_t nodeIndex = 0;

        while (true)
        {
            const PistonIFNode& node =
                nodes[
                    treeStart
                    + nodeIndex
                ];

            // leaf reached
            if (node.feature < 0)
            {
                depthSum +=
                    node.leafPath;

                break;
            }

            float value =
                featureValues[
                    node.feature
                ];

            // use the same median fill as Python
            if (isnan(value))
            {
                value =
                    imputerMedians[
                        node.feature
                    ];
            }

            if (value <= node.threshold)
            {
                nodeIndex =
                    node.left;
            }
            else
            {
                nodeIndex =
                    node.right;
            }
        }
    }

    float anomalyScore =
        powf(
            2.0f,
            -depthSum
            /
            normalization
        );

    return (
        anomalyScore
        +
        modelOffset
    );
}


static inline bool pistonIsolationForestAlert(
    float score,
    float threshold)
{
    return score >= threshold;
}
'''


with open(
    EXPORT_DIR / "piston_iforest_runtime.h",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        runtime_header
    )


# ============================================================
# EXPORT ONE MODEL
# ============================================================

def export_model_header(
    subsystem,
    model,
    imputer,
    features,
    piston_threshold,
    filename,
    extra_constants=None
):

    nodes, tree_offsets = (
        flatten_isolation_forest(
            model
        )
    )

    prefix = subsystem.upper()

    tree_count = len(
        model.estimators_
    )

    max_samples = int(
        model.max_samples_
    )

    normalization = (
        tree_count
        *
        average_path_length(
            max_samples
        )
    )

    model_offset = float(
        model.offset_
    )

    lines = [
        "#pragma once",
        "",
        '#include "piston_iforest_runtime.h"',
        "",
        f"// {subsystem} feature order",
        f"static const char* {prefix}_FEATURE_NAMES[] = {{",
    ]

    for feature in features:
        lines.append(
            f'    "{feature}",'
        )

    lines += [
        "};",
        "",
        f"static const float {prefix}_IMPUTER_MEDIANS[] = {{",
    ]

    for value in imputer.statistics_:
        lines.append(
            f"    {cpp_float(value)},"
        )

    lines += [
        "};",
        "",
        (
            f"static const uint8_t "
            f"{prefix}_FEATURE_COUNT = "
            f"{len(features)};"
        ),
        (
            f"static const uint16_t "
            f"{prefix}_TREE_COUNT = "
            f"{tree_count};"
        ),
        (
            f"static const float "
            f"{prefix}_NORMALIZATION = "
            f"{cpp_float(normalization)};"
        ),
        (
            f"static const float "
            f"{prefix}_MODEL_OFFSET = "
            f"{cpp_float(model_offset)};"
        ),
        (
            f"static const float "
            f"{prefix}_ANOMALY_THRESHOLD = "
            f"{cpp_float(piston_threshold)};"
        ),
        "",
    ]

    if extra_constants:

        lines.extend(
            extra_constants
        )

        lines.append("")

    lines.append(
        f"static const uint32_t "
        f"{prefix}_TREE_OFFSETS[] = {{"
    )

    for offset in tree_offsets:
        lines.append(
            f"    {offset},"
        )

    lines += [
        "};",
        "",
        (
            f"static const PistonIFNode "
            f"{prefix}_NODES[] = {{"
        ),
    ]

    for node in nodes:

        lines.append(
            "    {"
            f"{node['left']}, "
            f"{node['right']}, "
            f"{node['feature']}, "
            f"{cpp_float(node['threshold'])}, "
            f"{cpp_float(node['leaf_path'])}"
            "},"
        )

    lines += [
        "};",
        "",
    ]

    with open(
        EXPORT_DIR / filename,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(
            "\n".join(lines)
        )

    return {
        "subsystem":
            subsystem,

        "feature_count":
            len(features),

        "tree_count":
            tree_count,

        "node_count":
            len(nodes),

        "max_samples":
            max_samples,

        "normalization":
            normalization,

        "model_offset":
            model_offset,

        "piston_threshold":
            float(
                piston_threshold
            ),

        "features":
            list(features),

        "imputer_medians":
            [
                float(value)
                for value
                in imputer.statistics_
            ],
    }


# ============================================================
# FUEL MODEL
# ============================================================

fuel_export_info = export_model_header(
    subsystem="Fuel",
    model=fuel_model,
    imputer=fuel_imputer,
    features=FUEL_FEATURES,
    piston_threshold=FUEL_IF_THRESHOLD,
    filename="piston_fuel_model.h",
)


# ============================================================
# CATALYST MODEL
# ============================================================

catalyst_export_info = export_model_header(
    subsystem="Catalyst",
    model=catalyst_model,
    imputer=catalyst_imputer,
    features=CATALYST_FEATURES,
    piston_threshold=CATALYST_IF_THRESHOLD,
    filename="piston_catalyst_model.h",
    extra_constants=[
        (
            "static const float "
            "CATALYST_CORRELATION_THRESHOLD = "
            f"{cpp_float(CATALYST_CORRELATION_THRESHOLD)};"
        ),
        (
            "static const uint8_t "
            "CATALYST_REQUIRED_CONSECUTIVE_CASES = "
            f"{CATALYST_REQUIRED_CONSECUTIVE_CASES};"
        ),
    ],
)


# ============================================================
# CHARGING MODEL
# ============================================================

charging_export_info = export_model_header(
    subsystem="Charging",
    model=charging_model,
    imputer=charging_imputer,
    features=CHARGING_FEATURES,
    piston_threshold=CHARGING_IF_THRESHOLD,
    filename="piston_charging_model.h",
)


# ============================================================
# MODEL MANIFEST
# ============================================================

manifest = {
    "project":
        "P.I.S.T.O.N.",

    "version":
        "V4",

    "score_definition":
        "-model.decision_function(X)",

    "fuel":
        fuel_export_info,

    "catalyst":
        {
            **catalyst_export_info,

            "correlation_threshold":
                float(
                    CATALYST_CORRELATION_THRESHOLD
                ),

            "required_consecutive_cases":
                int(
                    CATALYST_REQUIRED_CONSECUTIVE_CASES
                ),
        },

    "charging":
        charging_export_info,
}


with open(
    EXPORT_DIR / "piston_model_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=4
    )


# ============================================================
# REBUILD TEST DATAFRAMES
#
# Do not depend on the fuel_test or charging_test
# variable names used by the display cell.
# ============================================================

fuel_export_test = (
    fuel_cases[
        fuel_cases[
            "DATA_SPLIT"
        ] == "TEST"
    ]
    .copy()
)


charging_export_test = (
    charging_cases[
        charging_cases[
            "DATA_SPLIT"
        ] == "TEST"
    ]
    .copy()
)


catalyst_export_test = (
    catalyst_cases[
        (
            catalyst_cases[
                "DATA_SPLIT"
            ] == "TEST"
        )
        &
        (
            catalyst_cases[
                "CATALYST_SIGNAL_EVALUABLE"
            ] == True
        )
    ]
    .copy()
)


X_fuel_export_test = (
    fuel_imputer.transform(
        fuel_export_test[
            FUEL_FEATURES
        ]
    )
)


X_charging_export_test = (
    charging_imputer.transform(
        charging_export_test[
            CHARGING_FEATURES
        ]
    )
)


X_catalyst_export_test = (
    catalyst_imputer.transform(
        catalyst_export_test[
            CATALYST_FEATURES
        ]
    )
)


# ============================================================
# TEST VECTOR EXPORT
# ============================================================

def save_test_vectors(
    data,
    transformed_features,
    model,
    threshold,
    feature_names,
    filename
):

    scores = (
        -model.decision_function(
            transformed_features
        )
    )


    metadata_columns = [
        column
        for column in [
            "CASE_ID",
            "VEHICLE_ID",
            "SOURCE_FILE",
            "DATA_SPLIT",
            "TARGET_ABNORMAL",
            "CASE_BEHAVIOR_LABEL",
        ]
        if column in data.columns
    ]


    output = (
        data[
            metadata_columns
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )


    for index, feature in enumerate(
        feature_names
    ):

        output[
            feature
        ] = (
            transformed_features[
                :,
                index
            ]
        )


    output[
        "EXPECTED_PISTON_SCORE"
    ] = scores


    output[
        "EXPECTED_IF_ALERT"
    ] = (
        scores
        >= threshold
    ).astype(int)


    output.to_csv(
        EXPORT_DIR / filename,
        index=False
    )


save_test_vectors(
    data=fuel_export_test,
    transformed_features=X_fuel_export_test,
    model=fuel_model,
    threshold=FUEL_IF_THRESHOLD,
    feature_names=FUEL_FEATURES,
    filename="fuel_test_vectors.csv",
)


save_test_vectors(
    data=catalyst_export_test,
    transformed_features=X_catalyst_export_test,
    model=catalyst_model,
    threshold=CATALYST_IF_THRESHOLD,
    feature_names=CATALYST_FEATURES,
    filename="catalyst_test_vectors.csv",
)


save_test_vectors(
    data=charging_export_test,
    transformed_features=X_charging_export_test,
    model=charging_model,
    threshold=CHARGING_IF_THRESHOLD,
    feature_names=CHARGING_FEATURES,
    filename="charging_test_vectors.csv",
)


# ============================================================
# EXPORT SUMMARY
# ============================================================

print("=" * 105)
print("ESP32 MODEL EXPORT COMPLETE")
print("=" * 105)


for info in [
    fuel_export_info,
    catalyst_export_info,
    charging_export_info,
]:

    print(
        f"\n{info['subsystem']}"
    )

    print(
        "  Features:",
        info[
            "feature_count"
        ]
    )

    print(
        "  Trees:",
        info[
            "tree_count"
        ]
    )

    print(
        "  Exported nodes:",
        info[
            "node_count"
        ]
    )

    print(
        "  Model offset:",
        round(
            info[
                "model_offset"
            ],
            6
        )
    )

    print(
        "  P.I.S.T.O.N. threshold:",
        round(
            info[
                "piston_threshold"
            ],
            6
        )
    )


print(
    "\nTest vectors:"
)

print(
    "  Fuel:",
    len(fuel_export_test)
)

print(
    "  Catalyst:",
    len(catalyst_export_test)
)

print(
    "  Charging:",
    len(charging_export_test)
)


print(
    "\nExport folder:",
    EXPORT_DIR.resolve()
)


print(
    "\nFiles created:"
)


for file in sorted(
    EXPORT_DIR.iterdir()
):

    print(
        "-",
        file.name
    )

ESP32 MODEL EXPORT COMPLETE

Fuel
  Features: 9
  Trees: 300
  Exported nodes: 39166
  Model offset: -0.5
  P.I.S.T.O.N. threshold: 0.080853

Catalyst
  Features: 5
  Trees: 300
  Exported nodes: 23800
  Model offset: -0.5
  P.I.S.T.O.N. threshold: 0.113591

Charging
  Features: 7
  Trees: 300
  Exported nodes: 36764
  Model offset: -0.5
  P.I.S.T.O.N. threshold: 0.146576

Test vectors:
  Fuel: 310
  Catalyst: 201
  Charging: 312

Export folder: C:\Users\yoboy\Documents\PISTONS_ML_V4\ESP32_Model_Export

Files created:
- catalyst_test_vectors.csv
- charging_test_vectors.csv
- fuel_test_vectors.csv
- piston_catalyst_model.h
- piston_charging_model.h
- piston_fuel_model.h
- piston_iforest_runtime.h
- piston_model_manifest.json


## 9. ESP32 Export Verification

Verify that the exported Isolation Forest representation reproduces the final Python model results.

In [10]:
# ============================================================
# ESP32 EXPORT VERIFICATION
#
# Run the exported tree representation using the same logic
# that will be used in C++ and compare it against scikit-learn.
# ============================================================


def exported_if_score(
    model,
    feature_values
):

    nodes, tree_offsets = (
        flatten_isolation_forest(
            model
        )
    )


    tree_count = len(
        model.estimators_
    )


    normalization = (
        tree_count
        *
        average_path_length(
            int(
                model.max_samples_
            )
        )
    )


    depth_sum = 0.0


    for tree_start in tree_offsets:

        node_index = 0


        while True:

            node = nodes[
                tree_start
                +
                node_index
            ]


            # leaf
            if node[
                "feature"
            ] < 0:

                depth_sum += (
                    node[
                        "leaf_path"
                    ]
                )

                break


            value = feature_values[
                node[
                    "feature"
                ]
            ]


            if (
                value
                <=
                node[
                    "threshold"
                ]
            ):

                node_index = (
                    node[
                        "left"
                    ]
                )

            else:

                node_index = (
                    node[
                        "right"
                    ]
                )


    anomaly_score = (
        2.0
        **
        (
            -depth_sum
            /
            normalization
        )
    )


    # same score definition used by P.I.S.T.O.N.
    piston_score = (
        anomaly_score
        +
        float(
            model.offset_
        )
    )


    return piston_score


# ============================================================
# VERIFY ONE SUBSYSTEM
# ============================================================

def verify_export(
    subsystem,
    model,
    features,
    threshold
):

    sklearn_scores = (
        -model.decision_function(
            features
        )
    )


    exported_scores = np.array(
        [
            exported_if_score(
                model,
                row
            )
            for row
            in features
        ]
    )


    sklearn_alerts = (
        sklearn_scores
        >= threshold
    ).astype(int)


    exported_alerts = (
        exported_scores
        >= threshold
    ).astype(int)


    score_error = np.abs(
        sklearn_scores
        -
        exported_scores
    )


    alert_matches = (
        sklearn_alerts
        ==
        exported_alerts
    )


    return {
        "SUBSYSTEM":
            subsystem,

        "CASES":
            len(features),

        "MAX_SCORE_ERROR":
            score_error.max(),

        "MEAN_SCORE_ERROR":
            score_error.mean(),

        "ALERT_MATCHES":
            int(
                alert_matches.sum()
            ),

        "ALERT_MISMATCHES":
            int(
                (
                    ~alert_matches
                ).sum()
            ),

        "ALERT_MATCH_RATE":
            alert_matches.mean(),
    }


# ============================================================
# RUN VERIFICATION
# ============================================================

verification_results = pd.DataFrame(
    [
        verify_export(
            "Fuel",
            fuel_model,
            X_fuel_export_test,
            FUEL_IF_THRESHOLD
        ),

        verify_export(
            "Catalyst",
            catalyst_model,
            X_catalyst_export_test,
            CATALYST_IF_THRESHOLD
        ),

        verify_export(
            "Charging",
            charging_model,
            X_charging_export_test,
            CHARGING_IF_THRESHOLD
        ),
    ]
)


print("=" * 115)
print("ESP32 EXPORTED MODEL VERIFICATION")
print("=" * 115)

print(
    verification_results
    .to_string(
        index=False
    )
)


# ============================================================
# PASS / FAIL
# ============================================================

total_mismatches = int(
    verification_results[
        "ALERT_MISMATCHES"
    ].sum()
)


if total_mismatches == 0:

    print(
        "\nPASS: All exported model alerts match "
        "the Python V4 models."
    )

else:

    print(
        "\nFAIL:",
        total_mismatches,
        "exported alerts do not match Python."
    )

ESP32 EXPORTED MODEL VERIFICATION
SUBSYSTEM  CASES  MAX_SCORE_ERROR  MEAN_SCORE_ERROR  ALERT_MATCHES  ALERT_MISMATCHES  ALERT_MATCH_RATE
     Fuel    310     1.110223e-16      2.148819e-18            310                 0               1.0
 Catalyst    201     0.000000e+00      0.000000e+00            201                 0               1.0
 Charging    312     0.000000e+00      0.000000e+00            312                 0               1.0

PASS: All exported model alerts match the Python V4 models.


In [11]:
# ============================================================
# FINAL ACTION LEVEL VERIFICATION
# Mirrors the final ESP32 action logic.
# Does not retrain or change any model.
# ============================================================

import pandas as pd

ACTION_LEVELS = [
    "Normal Operation",
    "Keep an Eye Out",
    "Inspect Soon",
]

# Fuel
FUEL_INSPECT_SCORE_THRESHOLD = 0.18
FUEL_KEEP_EYE_TRIM_THRESHOLD = 10.0
FUEL_INSPECT_TRIM_THRESHOLD = 18.0

# Catalyst
CATALYST_INSPECT_SCORE_THRESHOLD = 0.18
CATALYST_STRONG_CORRELATION_THRESHOLD = 0.90
CATALYST_STRONG_REQUIRED_CONSECUTIVE_CASES = 3

# Charging
CHARGING_KEEP_LOW_COUNT = 2
CHARGING_KEEP_HIGH_COUNT = 3
CHARGING_KEEP_MAX_STEP = 0.80
CHARGING_KEEP_STEP_STD = 0.30

CHARGING_INSPECT_LOW_MEAN = 13.0
CHARGING_INSPECT_HIGH_MEAN = 14.8
CHARGING_INSPECT_LOW_COUNT = 10
CHARGING_INSPECT_HIGH_COUNT = 5
CHARGING_INSPECT_MAX_STEP = 1.00
CHARGING_INSPECT_STEP_STD = 0.45
CHARGING_INSPECT_REQUIRED_CONSECUTIVE_CASES = 2


def print_action_table(title, data, label_column):

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    table = pd.crosstab(
        [
            data["DATA_SPLIT"],
            data[label_column],
        ],
        data["ACTION_LEVEL"]
    ).reindex(
        columns=ACTION_LEVELS,
        fill_value=0
    )

    print(table.to_string())


# ============================================================
# FUEL
# ============================================================

fuel_action = fuel_cases[
    fuel_cases["DATA_SPLIT"].isin(
        [
            "VALIDATION",
            "TEST",
        ]
    )
].copy()


fuel_action["FUEL_IF_SCORE"] = (
    -fuel_model.decision_function(
        fuel_imputer.transform(
            fuel_action[
                FUEL_FEATURES
            ]
        )
    )
)


fuel_trim_magnitude = (
    fuel_action[
        "SELECTED_BANK_TOTAL_TRIM_PCT_MEAN"
    ]
    .abs()
)


fuel_keep_evidence = (
    (
        fuel_action[
            "FUEL_IF_SCORE"
        ]
        >= FUEL_IF_THRESHOLD
    )
    |
    (
        fuel_trim_magnitude
        >= FUEL_KEEP_EYE_TRIM_THRESHOLD
    )
)


fuel_inspect_evidence = (
    (
        fuel_trim_magnitude
        >= FUEL_INSPECT_TRIM_THRESHOLD
    )
    |
    (
        (
            fuel_action[
                "FUEL_IF_SCORE"
            ]
            >= FUEL_INSPECT_SCORE_THRESHOLD
        )
        &
        (
            fuel_trim_magnitude
            >= FUEL_KEEP_EYE_TRIM_THRESHOLD
        )
    )
)


fuel_action[
    "ACTION_LEVEL"
] = "Normal Operation"


fuel_action.loc[
    fuel_keep_evidence,
    "ACTION_LEVEL"
] = "Keep an Eye Out"


fuel_action.loc[
    fuel_inspect_evidence,
    "ACTION_LEVEL"
] = "Inspect Soon"


print_action_table(
    "FUEL ACTION LEVEL VERIFICATION",
    fuel_action,
    "TARGET_ABNORMAL"
)


# ============================================================
# CATALYST
# ============================================================

catalyst_action = (
    catalyst_cases[
        catalyst_cases[
            "DATA_SPLIT"
        ].isin(
            [
                "VALIDATION",
                "TEST",
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


catalyst_action[
    "CATALYST_IF_SCORE"
] = float("nan")


catalyst_action[
    "PERSISTENT_MIRROR"
] = False


catalyst_action[
    "STRONG_PERSISTENT_MIRROR"
] = False


catalyst_action[
    "ACTION_LEVEL"
] = "Not Evaluated"


catalyst_evaluable_mask = (
    catalyst_action[
        "CATALYST_SIGNAL_EVALUABLE"
    ]
    .astype(bool)
)


catalyst_action.loc[
    catalyst_evaluable_mask,
    "CATALYST_IF_SCORE"
] = (
    -catalyst_model.decision_function(
        catalyst_imputer.transform(
            catalyst_action.loc[
                catalyst_evaluable_mask,
                CATALYST_FEATURES
            ]
        )
    )
)


for _, group in (
    catalyst_action
    .sort_values(
        [
            "SOURCE_FILE",
            "START_SAMPLE",
        ]
    )
    .groupby(
        "SOURCE_FILE",
        dropna=False
    )
):

    mirror_streak = 0
    strong_mirror_streak = 0


    for index, row in group.iterrows():

        # Incomplete Catalyst cases leave the streak unchanged
        if not bool(
            row[
                "CATALYST_SIGNAL_EVALUABLE"
            ]
        ):
            continue


        correlation = pd.to_numeric(
            row[
                "CAT_MAX_LAGGED_CORR"
            ],
            errors="coerce"
        )


        mirroring_this_case = (
            pd.notna(
                correlation
            )
            and
            correlation
            >= CATALYST_CORRELATION_THRESHOLD
        )


        strong_mirroring_this_case = (
            pd.notna(
                correlation
            )
            and
            correlation
            >= CATALYST_STRONG_CORRELATION_THRESHOLD
        )


        if mirroring_this_case:

            mirror_streak += 1

        else:

            mirror_streak = 0


        if strong_mirroring_this_case:

            strong_mirror_streak += 1

        else:

            strong_mirror_streak = 0


        if (
            mirror_streak
            >= CATALYST_REQUIRED_CONSECUTIVE_CASES
        ):

            catalyst_action.loc[
                index,
                "PERSISTENT_MIRROR"
            ] = True


        if (
            strong_mirror_streak
            >= CATALYST_STRONG_REQUIRED_CONSECUTIVE_CASES
        ):

            catalyst_action.loc[
                index,
                "STRONG_PERSISTENT_MIRROR"
            ] = True


for index, row in catalyst_action.iterrows():

    if not bool(
        row[
            "CATALYST_SIGNAL_EVALUABLE"
        ]
    ):
        continue


    catalyst_keep_evidence = (
        row[
            "CATALYST_IF_SCORE"
        ]
        >= CATALYST_IF_THRESHOLD
        or
        bool(
            row[
                "PERSISTENT_MIRROR"
            ]
        )
    )


    catalyst_inspect_evidence = (
        (
            row[
                "CATALYST_IF_SCORE"
            ]
            >= CATALYST_INSPECT_SCORE_THRESHOLD
            and
            bool(
                row[
                    "PERSISTENT_MIRROR"
                ]
            )
        )
        or
        bool(
            row[
                "STRONG_PERSISTENT_MIRROR"
            ]
        )
    )


    if catalyst_inspect_evidence:

        catalyst_action.loc[
            index,
            "ACTION_LEVEL"
        ] = "Inspect Soon"


    elif catalyst_keep_evidence:

        catalyst_action.loc[
            index,
            "ACTION_LEVEL"
        ] = "Keep an Eye Out"


    else:

        catalyst_action.loc[
            index,
            "ACTION_LEVEL"
        ] = "Normal Operation"


catalyst_evaluable = (
    catalyst_action[
        catalyst_action[
            "CATALYST_SIGNAL_EVALUABLE"
        ].astype(bool)
    ]
    .copy()
)


print_action_table(
    "CATALYST ACTION LEVEL VERIFICATION",
    catalyst_evaluable,
    "CASE_BEHAVIOR_LABEL"
)


print(
    "\nNon-evaluable Catalyst cases:",
    int(
        (
            ~catalyst_action[
                "CATALYST_SIGNAL_EVALUABLE"
            ]
            .astype(bool)
        ).sum()
    )
)


# ============================================================
# CHARGING
# ============================================================

charging_action = (
    charging_cases[
        charging_cases[
            "DATA_SPLIT"
        ].isin(
            [
                "VALIDATION",
                "TEST",
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


charging_action[
    "CHARGING_IF_SCORE"
] = (
    -charging_model.decision_function(
        charging_imputer.transform(
            charging_action[
                CHARGING_FEATURES
            ]
        )
    )
)


charging_action[
    "CHARGING_IF_ALERT"
] = (
    charging_action[
        "CHARGING_IF_SCORE"
    ]
    >= CHARGING_IF_THRESHOLD
)


charging_low_mild = (
    charging_action[
        "BELOW_CHARGING_COUNT_LT_13_0"
    ]
    >= CHARGING_KEEP_LOW_COUNT
)


charging_high_mild = (
    charging_action[
        "ABOVE_CHARGING_COUNT_GT_14_8"
    ]
    >= CHARGING_KEEP_HIGH_COUNT
)


charging_step_mild = (
    charging_action[
        "VOLTAGE_MAX_ABS_STEP"
    ]
    >= CHARGING_KEEP_MAX_STEP
)


charging_step_std_mild = (
    charging_action[
        "VOLTAGE_STEP_STD"
    ]
    >= CHARGING_KEEP_STEP_STD
)


charging_action[
    "KEEP_EVIDENCE"
] = (
    charging_action[
        "CHARGING_IF_ALERT"
    ]
    |
    charging_low_mild
    |
    charging_high_mild
    |
    charging_step_mild
    |
    charging_step_std_mild
)


charging_action[
    "STRONG_PHYSICAL_THIS_CASE"
] = (
    (
        charging_action[
            "CONTROL_MODULE_VOLTAGE_V_MEAN"
        ]
        < CHARGING_INSPECT_LOW_MEAN
    )
    |
    (
        charging_action[
            "CONTROL_MODULE_VOLTAGE_V_MEAN"
        ]
        > CHARGING_INSPECT_HIGH_MEAN
    )
    |
    (
        charging_action[
            "BELOW_CHARGING_COUNT_LT_13_0"
        ]
        >= CHARGING_INSPECT_LOW_COUNT
    )
    |
    (
        charging_action[
            "ABOVE_CHARGING_COUNT_GT_14_8"
        ]
        >= CHARGING_INSPECT_HIGH_COUNT
    )
    |
    (
        charging_action[
            "VOLTAGE_MAX_ABS_STEP"
        ]
        >= CHARGING_INSPECT_MAX_STEP
    )
    |
    (
        charging_action[
            "VOLTAGE_STEP_STD"
        ]
        >= CHARGING_INSPECT_STEP_STD
    )
)


charging_action[
    "STRONG_STREAK"
] = 0


for _, group in (
    charging_action
    .sort_values(
        [
            "SOURCE_FILE",
            "START_SAMPLE",
        ]
    )
    .groupby(
        "SOURCE_FILE",
        dropna=False
    )
):

    strong_streak = 0


    for index, row in group.iterrows():

        if bool(
            row[
                "STRONG_PHYSICAL_THIS_CASE"
            ]
        ):

            strong_streak += 1

        else:

            strong_streak = 0


        charging_action.loc[
            index,
            "STRONG_STREAK"
        ] = strong_streak


charging_action[
    "ACTION_LEVEL"
] = "Normal Operation"


charging_action.loc[
    charging_action[
        "KEEP_EVIDENCE"
    ],
    "ACTION_LEVEL"
] = "Keep an Eye Out"


charging_action.loc[
    charging_action[
        "STRONG_STREAK"
    ]
    >= CHARGING_INSPECT_REQUIRED_CONSECUTIVE_CASES,
    "ACTION_LEVEL"
] = "Inspect Soon"


print_action_table(
    "CHARGING ACTION LEVEL VERIFICATION",
    charging_action,
    "TARGET_ABNORMAL"
)


# ============================================================
# HEALTHY INSPECT SOON CHECK
# ============================================================

fuel_healthy_inspect = (
    (
        fuel_action[
            "TARGET_ABNORMAL"
        ] == 0
    )
    &
    (
        fuel_action[
            "ACTION_LEVEL"
        ] == "Inspect Soon"
    )
).sum()


catalyst_healthy_inspect = (
    (
        catalyst_action[
            "CASE_BEHAVIOR_LABEL"
        ] == "NORMAL"
    )
    &
    (
        catalyst_action[
            "CATALYST_SIGNAL_EVALUABLE"
        ]
        .astype(bool)
    )
    &
    (
        catalyst_action[
            "ACTION_LEVEL"
        ] == "Inspect Soon"
    )
).sum()


charging_healthy_inspect = (
    (
        charging_action[
            "TARGET_ABNORMAL"
        ] == 0
    )
    &
    (
        charging_action[
            "ACTION_LEVEL"
        ] == "Inspect Soon"
    )
).sum()


print("\n" + "=" * 100)
print("HEALTHY INSPECT SOON CHECK")
print("=" * 100)


print(
    "Fuel healthy Inspect Soon cases:",
    int(
        fuel_healthy_inspect
    )
)


print(
    "Catalyst healthy Inspect Soon cases:",
    int(
        catalyst_healthy_inspect
    )
)


print(
    "Charging healthy Inspect Soon cases:",
    int(
        charging_healthy_inspect
    )
)


FUEL ACTION LEVEL VERIFICATION
ACTION_LEVEL                Normal Operation  Keep an Eye Out  Inspect Soon
DATA_SPLIT TARGET_ABNORMAL                                                 
TEST       0                             193               14             0
           1                               8               94             1
VALIDATION 0                              43                6             0
           1                               1               42            14

CATALYST ACTION LEVEL VERIFICATION
ACTION_LEVEL                    Normal Operation  Keep an Eye Out  Inspect Soon
DATA_SPLIT CASE_BEHAVIOR_LABEL                                                 
TEST       NORMAL                            189               12             0
VALIDATION NORMAL                             12                1             0
           UNCERTAIN                          91                2             0

Non-evaluable Catalyst cases: 36

CHARGING ACTION LEVEL VERIFICATION
ACTION